<a href="https://colab.research.google.com/github/achinthyam08/AI-LAB/blob/main/1BF24CS014_Week6_FOL_Unification_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import re

# Variables used in this lab: x, y, z, u, v, w
VARIABLES = {"x", "y", "z", "u", "v", "w"}


# Convert an expression into tokens
def tokenize(expression):
    return re.findall(
        r"[A-Za-z_][A-Za-z_0-9]*|[(),]",
        expression
    )


# Parse a term or predicate recursively
def parse_term(tokens, pos):
    if pos >= len(tokens):
        raise ValueError("Unexpected end of expression")

    name = tokens[pos]
    pos += 1

    if pos < len(tokens) and tokens[pos] == "(":
        pos += 1
        args = []

        if pos < len(tokens) and tokens[pos] == ")":
            raise ValueError("Empty argument list")

        while True:
            arg, pos = parse_term(tokens, pos)
            args.append(arg)

            if pos >= len(tokens):
                raise ValueError("Missing closing parenthesis")

            if tokens[pos] == ")":
                pos += 1
                break

            if tokens[pos] != ",":
                raise ValueError("Expected comma or closing parenthesis")

            pos += 1

        return (name, tuple(args)), pos

    return name, pos


def parse_expression(expression):
    tokens = tokenize(expression)

    if not tokens:
        raise ValueError("Empty expression")

    term, pos = parse_term(tokens, 0)

    if pos != len(tokens):
        raise ValueError("Invalid expression")

    return term


# Check whether a term is a variable
def is_variable(term):
    return isinstance(term, str) and term in VARIABLES


# Apply existing substitutions recursively
def apply_substitution(term, subst):
    if is_variable(term) and term in subst:
        return apply_substitution(subst[term], subst)

    if isinstance(term, tuple):
        name, args = term
        return (
            name,
            tuple(apply_substitution(arg, subst) for arg in args)
        )

    return term


# Occurs check: prevent x -> f(x)
def occurs(variable, term, subst):
    term = apply_substitution(term, subst)

    if variable == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(variable, arg, subst)
                   for arg in term[1])

    return False


# UNIFY algorithm
# Returns substitutions on success and None on failure
def unify(term1, term2, subst):
    term1 = apply_substitution(term1, subst)
    term2 = apply_substitution(term2, subst)

    # Step 1: Identical expressions need no new substitution
    if term1 == term2:
        return subst

    # Step 2: First expression is a variable
    if is_variable(term1):
        if occurs(term1, term2, subst):
            return None

        subst[term1] = term2
        return subst

    # Step 3: Second expression is a variable
    if is_variable(term2):
        if occurs(term2, term1, subst):
            return None

        subst[term2] = term1
        return subst

    # Step 4: Both are compound expressions
    if isinstance(term1, tuple) and isinstance(term2, tuple):
        name1, args1 = term1
        name2, args2 = term2

        # Predicate/function symbols must match
        if name1 != name2:
            return None

        # Number of arguments must match
        if len(args1) != len(args2):
            return None

        # Unify corresponding arguments recursively
        for arg1, arg2 in zip(args1, args2):
            if unify(arg1, arg2, subst) is None:
                return None

        return subst

    # Different constants cannot be unified
    return None


# Convert internal terms into readable expressions
def display(term):
    if isinstance(term, tuple):
        name, args = term
        return name + "(" + ", ".join(
            display(arg) for arg in args
        ) + ")"

    return term


def main():
    print("FIRST-ORDER LOGIC: UNIFICATION")
    print("--------------------------------")

    expression1 = input("Enter first expression: ")
    expression2 = input("Enter second expression: ")

    try:
        term1 = parse_expression(expression1)
        term2 = parse_expression(expression2)

        subst = unify(term1, term2, {})

        if subst is None:
            print("\nUnification failed.")
        else:
            print("\nUnification successful.")
            print("Most General Unifier (MGU):")

            if not subst:
                print("{}")
            else:
                for variable in subst:
                    value = apply_substitution(
                        subst[variable], subst
                    )
                    print(variable, "->", display(value))

            print("\nUnified expression:")
            print(display(apply_substitution(term1, subst)))

    except ValueError as error:
        print("Invalid expression:", error)


if __name__ == "__main__":
    main()


FIRST-ORDER LOGIC: UNIFICATION
--------------------------------
Enter first expression: p(x,f(y))
Enter second expression: p(a,f(b))

Unification successful.
Most General Unifier (MGU):
x -> a
y -> b

Unified expression:
p(a, f(b))
